# Lab: Perturbing AI weather models (Earth2Studio)

Purely data-driven weather models learn to step the atmosphere forward by training on decades of ERA5 reanalysis. They have no equations of motion inside them. NVIDIA's **Earth2Studio** gives several of these models the same interface, so you can run the *same* experiment in different models and compare them.

| model | developer | grid | time step | architecture |
|---|---|---|---|---|
| `Pangu24` | Huawei (2023) | 0.25°, 13 levels | 24 h | 3-D Earth-specific transformer |
| `Pangu6` | Huawei (2023) | 0.25°, 13 levels | 6 h | same, 6 h + 24 h models interleaved |
| `FCN` | NVIDIA (2022) | 0.25°, 26 variables | 6 h | FourCastNet (Fourier neural operator) |
| `DLWP` | U. Washington / NVIDIA | cubed sphere, 7 variables | 6 h | convolutional network |

In this lab you will:
1. Run a 10-day forecast and compare it with ERA5.
2. Test the **butterfly effect**: does a tiny temperature perturbation grow?
3. Add a **localized warm anomaly** and follow the response.
4. **Compare models** on the same experiment.
5. Design **your own experiment**.

### Before you start
* `Runtime → Change runtime type → GPU`. A free T4 (16 GB) should work for these models, and an L4 or A100 is faster. If you run out of GPU memory, switch to `FCN` or `DLWP`.
* Installing takes a few minutes. If Colab asks you to **restart the session** after installing, do so, then continue from the import cell.
* The first time you use each model it downloads its weights (~1–2 GB).
* Cells marked **✏️** are the ones you are meant to edit.

In [ ]:
#@title Install Earth2Studio with the Pangu, FourCastNet and DLWP models (~3–5 min)
!pip install -q "earth2studio[pangu,fcn,dlwp]==0.19.0" cartopy

In [ ]:
#@title Imports and GPU check
import gc
import re
import time
from datetime import datetime, timedelta

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import torch
import xarray as xr

import earth2studio.models.px as px
from earth2studio.data import ARCO, fetch_data
from earth2studio.utils.coords import map_coords
from earth2studio.utils.time import to_time_array

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}, {props.total_memory / 1e9:.0f} GB")
else:
    print("*** No GPU found. Use Runtime -> Change runtime type -> GPU, then rerun. ***")

try:
    import onnxruntime as ort
    if "CUDAExecutionProvider" not in ort.get_available_providers():
        print("Note: ONNX Runtime cannot see the GPU, so Pangu will run on the CPU (slow).")
except ImportError:
    pass

plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "figure.dpi": 110})
G = 9.80665  # m s^-2, converts geopotential to geopotential height

In [ ]:
#@title Helper functions: loading models, running forecasts, diagnostics, maps
data = ARCO()  # ERA5 from Google's public ARCO-ERA5 store (1940 to 2025)
SAVE_VARS = ["z500", "t850", "t2m", "u250", "msl"]  # saved when the model has them
_loaded = {"name": None, "model": None}


def get_model(name):
    # Load an Earth2Studio prognostic model by class name, freeing the previous one
    if _loaded["name"] != name:
        _loaded["model"] = None
        gc.collect()
        torch.cuda.empty_cache()
        cls = getattr(px, name)
        print(f"Loading {name} (downloads weights the first time)...")
        _loaded["model"] = cls.load_model(cls.load_default_package()).to(device)
        _loaded["name"] = name
    return _loaded["model"]


def initial_condition(model, start_time):
    # ERA5 state on the model's grid, as (tensor, coords)
    ic = model.input_coords()
    x, coords = fetch_data(source=data, time=to_time_array([start_time]),
                           variable=ic["variable"], lead_time=ic["lead_time"], device=device)
    return map_coords(x, coords, ic)


@torch.inference_mode()
def run_forecast(model, x, coords, days, save_vars=SAVE_VARS):
    # Integrate `model` from (x, coords) for `days` days.
    # Returns an xarray Dataset of the saved variables with dimension `lead_day`.
    step_h = model.output_coords(model.input_coords())["lead_time"][-1] / np.timedelta64(1, "h")
    nsteps = int(round(days * 24 / step_h))
    names = list(coords["variable"])
    keep = [v for v in save_vars if v in names]
    idx = [names.index(v) for v in keep]
    frames, leads = [], []
    for step, (y, yc) in enumerate(model.create_iterator(x.clone(), coords.copy())):
        arr = y[..., idx, :, :].float().cpu().numpy()
        frames.append(arr.reshape(-1, *arr.shape[-3:])[-1])  # latest time -> (variable, lat, lon)
        leads.append(yc["lead_time"][-1] / np.timedelta64(1, "h") / 24)
        if step == nsteps:
            break
    da = xr.DataArray(np.stack(frames), dims=("lead_day", "variable", "lat", "lon"),
                      coords={"lead_day": leads, "variable": keep, "lat": yc["lat"], "lon": yc["lon"]})
    return da.to_dataset("variable")


def era5_truth(start_time, days, like, variables=("z500", "t850")):
    # Daily ERA5 at the forecast's valid times, on the same grid as forecast `like`
    t0 = datetime.fromisoformat(start_time)
    times = [t0 + timedelta(days=d) for d in range(days + 1)]
    da = data(times, list(variables)).sel(lat=like.lat, lon=like.lon, method="nearest")
    da = da.assign_coords(lat=like.lat.values, lon=like.lon.values)
    ds = da.to_dataset("variable").rename(time="lead_day")
    return ds.assign_coords(lead_day=np.arange(days + 1))


def area_rms(da):
    # cos(latitude)-weighted RMS over the globe
    w = np.cos(np.deg2rad(da.lat))
    return np.sqrt((da ** 2).weighted(w).mean(("lat", "lon")))


def var_index(coords, name):
    return list(coords["variable"]).index(name)


def temperature_vars(coords):
    # Pressure-level temperatures (t50 ... t1000) that the model uses
    return [v for v in coords["variable"] if re.fullmatch(r"t\d+", v)]


def gaussian_blob(coords, lat0, lon0, radius_km):
    # exp(-(d/R)^2) on the model grid as a torch tensor (lat, lon)
    lat = np.deg2rad(np.asarray(coords["lat"]))[:, None]
    lon = np.deg2rad(np.asarray(coords["lon"]))[None, :]
    la0, lo0 = np.deg2rad(lat0), np.deg2rad(lon0)
    cosd = np.sin(lat) * np.sin(la0) + np.cos(lat) * np.cos(la0) * np.cos(lon - lo0)
    dist_km = 6371.0 * np.arccos(np.clip(cosd, -1.0, 1.0))
    return torch.as_tensor(np.exp(-((dist_km / radius_km) ** 2)), dtype=torch.float32, device=device)


def add_temperature_noise(x, coords, sigma_K, seed=0):
    # Copy of x with N(0, sigma_K) noise added to every pressure-level temperature
    x = x.clone()
    gen = torch.Generator(device=x.device).manual_seed(seed)
    for v in temperature_vars(coords):
        i = var_index(coords, v)
        field = x[..., i, :, :]
        x[..., i, :, :] = field + sigma_K * torch.randn(field.shape, generator=gen, device=x.device)
    return x


def add_warm_blob(x, coords, lat0, lon0, radius_km, amplitude_K, levels_hPa):
    # Copy of x with a Gaussian warm anomaly added to t<level> for each level the model has
    x = x.clone()
    blob = amplitude_K * gaussian_blob(coords, lat0, lon0, radius_km)
    used = []
    for lev in levels_hPa:
        name = f"t{lev}"
        if name in list(coords["variable"]):
            x[..., var_index(coords, name), :, :] += blob
            used.append(name)
    print("Warm anomaly added to:", used or "nothing (this model has none of those levels)")
    return x


def _cyclic(da):
    first = da.isel(lon=[0])
    return xr.concat([da, first.assign_coords(lon=first.lon + 360)], "lon")


def plot_maps(fields, titles, cmap="RdBu_r", vmax=None, contours=None, cbar_label="",
              ncols=2, central_longitude=180):
    # Panel of global maps; `contours` is an optional matching list drawn as black lines
    n = len(fields)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.5 * ncols, 3.4 * nrows), squeeze=False,
                             subplot_kw={"projection": ccrs.Robinson(central_longitude)})
    if vmax is None:
        vmax = float(max(np.abs(f).max() for f in fields))
    diverging = cmap in ("RdBu_r", "BrBG", "PuOr_r", "coolwarm")
    vmin = -vmax if diverging else float(min(f.min() for f in fields))
    for i, ax in enumerate(axes.flat):
        if i >= n:
            ax.remove()
            continue
        f = _cyclic(fields[i])
        mesh = ax.pcolormesh(f.lon, f.lat, f, transform=ccrs.PlateCarree(), cmap=cmap,
                             vmin=vmin, vmax=vmax, shading="auto", rasterized=True)
        if contours is not None:
            c = _cyclic(contours[i])
            ax.contour(c.lon, c.lat, c, levels=12, colors="k", linewidths=0.6,
                       transform=ccrs.PlateCarree())
        ax.add_feature(cfeature.COASTLINE, linewidth=0.4, edgecolor="0.3")
        ax.set_global()
        ax.set_title(titles[i])
    fig.colorbar(mesh, ax=axes, orientation="horizontal", shrink=0.5, pad=0.04, label=cbar_label)
    plt.show()

---
## Setup: choose a model and a start date

The initial state is ERA5 at `START_TIME`. The default, **8 Feb 2021**, is just before the February 2021 cold-air outbreak over Texas. Any date from 1940 to 2025 works (00 UTC is safest).

In [ ]:
#@title ✏️ Model and start date
MODEL = "Pangu24"  #@param ["Pangu24", "Pangu6", "FCN", "DLWP"]
START_TIME = "2021-02-08T00:00"  #@param {type:"string"}

tic = time.time()
model = get_model(MODEL)
x0, coords0 = initial_condition(model, START_TIME)
print(f"Ready in {time.time() - tic:.0f} s")
print(f"{MODEL} uses {len(coords0['variable'])} variables:", ", ".join(coords0["variable"]))

---
## Part 1: Control forecast vs. ERA5

In [ ]:
#@title ✏️ Run the control forecast
FORECAST_DAYS = 10  #@param {type:"integer"}

tic = time.time()
control = run_forecast(model, x0, coords0, FORECAST_DAYS)
truth = era5_truth(START_TIME, FORECAST_DAYS, like=control)
print(f"Forecast done in {time.time() - tic:.0f} s")

daily = control.sel(lead_day=truth.lead_day)  # 6-hourly models: keep whole days
fields, titles = [], []
for d in [d for d in (1, 3, 5, FORECAST_DAYS) if d <= FORECAST_DAYS]:
    fields += [truth.z500.sel(lead_day=d) / G, daily.z500.sel(lead_day=d) / G]
    titles += [f"ERA5, day {d}", f"{MODEL}, day {d}"]
plot_maps(fields, titles, cmap="viridis", vmax=float(truth.z500.max() / G),
          cbar_label="500 hPa geopotential height (m)")

err = area_rms((daily.z500 - truth.z500) / G)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(err.lead_day, err, "o-", lw=2)
ax.set(xlabel="Lead time (days)", ylabel="Global RMS error (m)", title=f"Z500 error of {MODEL} vs. ERA5")
ax.grid(alpha=0.3)
plt.show()

**Questions**
1. When do the forecast and ERA5 start to differ clearly? Which features are still right at day 10?
2. Look closely at the day-10 forecast map. Does it look as sharp and detailed as ERA5? Why might a model trained to minimize mean-squared error produce smooth forecasts at long lead times?

---
## Part 2: The butterfly effect

Add random noise with a standard deviation of only `NOISE_K` kelvin to every temperature field, then measure how far the perturbed forecast drifts from the control. In the real atmosphere, and in physics-based models, tiny differences grow roughly exponentially until the two forecasts are as different as two random weather states.

In [ ]:
#@title ✏️ Perturbed forecast
NOISE_K = 0.01  #@param {type:"number"}
BUTTERFLY_DAYS = 15  #@param {type:"integer"}


def butterfly(model, x, coords, noise_K, days):
    ctrl = run_forecast(model, x, coords, days)
    pert = run_forecast(model, add_temperature_noise(x, coords, noise_K, seed=1), coords, days)
    return ctrl, pert


control_long, noisy = butterfly(model, x0, coords0, NOISE_K, BUTTERFLY_DAYS)
diff = (noisy.z500 - control_long.z500) / G
saturation = float(area_rms((control_long.z500.isel(lead_day=-1) - control_long.z500.isel(lead_day=0)) / G))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogy(diff.lead_day, area_rms(diff), "o-", lw=2, label=f"perturbed - control ({NOISE_K} K noise)")
ax.axhline(saturation, color="0.5", ls="--", label="~ difference between unrelated states")
ax.set(xlabel="Lead time (days)", ylabel="Global RMS Z500 difference (m)",
       title=f"{MODEL}: growth of a tiny perturbation")
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which="both")
plt.show()

show = [d for d in (1, 3, 6, 10) if d <= BUTTERFLY_DAYS]
plot_maps([diff.sel(lead_day=d) for d in show], [f"Z500 difference, day {d}" for d in show],
          contours=[control_long.z500.sel(lead_day=d) / G for d in show],
          cbar_label="Perturbed - control Z500 (m); contours: control Z500")

**Questions**
1. Does the difference grow? Estimate a doubling time, if there is one, and compare it with the ~1–2 day doubling time typical of synoptic-scale errors in physics-based models.
2. Try `NOISE_K = 0.001`, `0.1` and `1.0`. Is the growth rate the same for every amplitude?
3. Selz & Craig (2023) found that Pangu-Weather does not amplify very small perturbations the way physics-based models do, although it handles larger ones reasonably. Do your results agree? Why might a model trained on ERA5 lack this behavior?

---
## Part 3: A localized warm anomaly

Add a Gaussian warm anomaly to the temperature at several pressure levels and follow its evolution. Only temperature is changed: geopotential and winds are left alone, so the initial state is *not* hydrostatically or geostrophically balanced.

`DLWP` has only `t850` and `t2m`, so for this part `Pangu` or `FCN` work best.

In [ ]:
#@title ✏️ Warm anomaly experiment
BLOB_LAT = 40.0  #@param {type:"number"}
BLOB_LON = 180.0  #@param {type:"number"}
BLOB_RADIUS_KM = 1000.0  #@param {type:"number"}
BLOB_AMPLITUDE_K = 3.0  #@param {type:"number"}
BLOB_LEVELS_HPA = "1000, 925, 850, 700, 600, 500"  #@param {type:"string"}
BLOB_DAYS = 10  #@param {type:"integer"}

levels = [int(s) for s in BLOB_LEVELS_HPA.split(",")]
x_warm = add_warm_blob(x0, coords0, BLOB_LAT, BLOB_LON, BLOB_RADIUS_KM, BLOB_AMPLITUDE_K, levels)
warm = run_forecast(model, x_warm, coords0, BLOB_DAYS)
ctrl = control_long if BLOB_DAYS <= BUTTERFLY_DAYS else run_forecast(model, x0, coords0, BLOB_DAYS)
ctrl = ctrl.sel(lead_day=warm.lead_day)

show = [d for d in (0, 1, 2, 4, 6, 10) if d <= BLOB_DAYS]
plot_maps([(warm.t850 - ctrl.t850).sel(lead_day=d) for d in show],
          [f"T850 difference, day {d}" for d in show], cbar_label="Warm - control T850 (K)")
plot_maps([((warm.z500 - ctrl.z500) / G).sel(lead_day=d) for d in show],
          [f"Z500 difference, day {d}" for d in show],
          contours=[ctrl.z500.sel(lead_day=d) / G for d in show],
          cbar_label="Warm - control Z500 (m); contours: control Z500")

**Questions**
1. Look at day 0 vs. day 1. Does the Z500 response appear right away? What does the hypsometric equation say should happen?
2. How does the signal spread? Does it follow the jet, and does it travel faster than the wind?
3. Hakim & Masanam (2024) found that Pangu-Weather gives physically reasonable responses to localized heating (e.g. a Gill-like response in the tropics). Move the anomaly to the equator (`BLOB_LAT = 0`). What do you see?
4. Try a much larger amplitude (e.g. 20 K). Does the response still look physical? What happens when the model is pushed far outside the states it was trained on?

---
## Part 4: Same experiment, different models

Run the butterfly experiment and the control forecast in several models and compare them. Each model is loaded in turn, and the first load of each downloads its weights.

In [ ]:
#@title ✏️ Compare models
MODELS_TO_COMPARE = "Pangu24, FCN, DLWP"  #@param {type:"string"}
COMPARE_NOISE_K = 0.01  #@param {type:"number"}
COMPARE_DAYS = 10  #@param {type:"integer"}

results = {}
for name in [s.strip() for s in MODELS_TO_COMPARE.split(",")]:
    tic = time.time()
    m = get_model(name)
    x, c = initial_condition(m, START_TIME)
    ctrl, pert = butterfly(m, x, c, COMPARE_NOISE_K, COMPARE_DAYS)
    tr = era5_truth(START_TIME, COMPARE_DAYS, like=ctrl)
    results[name] = {
        "error": area_rms((ctrl.z500.sel(lead_day=tr.lead_day) - tr.z500) / G),
        "growth": area_rms((pert.z500 - ctrl.z500) / G),
        "day5": ctrl.z500.sel(lead_day=5) / G,
    }
    print(f"{name}: done in {time.time() - tic:.0f} s")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for name, r in results.items():
    ax1.plot(r["error"].lead_day, r["error"], "o-", lw=2, label=name)
    ax2.semilogy(r["growth"].lead_day, r["growth"], "o-", lw=2, label=name)
ax1.set(xlabel="Lead time (days)", ylabel="Global RMS error (m)", title="Z500 error vs. ERA5")
ax2.set(xlabel="Lead time (days)", ylabel="Global RMS difference (m)",
        title=f"Growth of {COMPARE_NOISE_K} K temperature noise")
for ax in (ax1, ax2):
    ax.legend()
    ax.grid(alpha=0.3, which="both")
plt.show()

names = list(results)
plot_maps([results[n]["day5"] for n in names], [f"{n}: Z500, day 5" for n in names],
          cmap="viridis", vmax=max(float(results[n]["day5"].max()) for n in names),
          cbar_label="500 hPa geopotential height (m)")

**Questions**
1. Which model has the smallest error at day 5? At day 10? Are the rankings the same at both lead times?
2. Do the models agree on how fast small perturbations grow? If not, which one looks most like the real atmosphere?
3. If two models make similarly good forecasts but respond very differently to perturbations, what does that say about using them for ensemble forecasting or for "what if" experiments?

---
## Part 5: Your own experiment

Design a perturbation, write down your prediction **before** you run it, then test it. Some ideas:

* Weaken or strengthen the jet: scale `u250` (and the other `u` levels) in a latitude band.
* Add a cold anomaly over the Arctic and look for a cold-air outbreak.
* Dry out a region: scale the `q` (specific humidity) variables in Pangu.
* Repeat Part 3 with geopotential adjusted to match the warm anomaly. Does balancing the initial state change the response?

The initial state is the tensor `x0` with coordinates `coords0`. Use `var_index(coords0, "u250")` to find a variable, modify a `x0.clone()`, and run it with `run_forecast(model, x, coords0, days)`.

In [ ]:
#@title ✏️ Example: weaken the Northern Hemisphere jet by 30% between 30–60°N
my_days = 10
lat = torch.as_tensor(np.asarray(coords0["lat"]), device=device, dtype=torch.float32)
band = ((lat >= 30) & (lat <= 60)).float()[:, None]  # (lat, 1), broadcasts over lon

x_mine = x0.clone()
for v in coords0["variable"]:
    if re.fullmatch(r"u\d+", v):  # all pressure-level zonal winds
        i = var_index(coords0, v)
        x_mine[..., i, :, :] *= 1 - 0.3 * band

mine = run_forecast(model, x_mine, coords0, my_days)
ctrl = run_forecast(model, x0, coords0, my_days)

show = [0, 2, 5, my_days]
plot_maps([((mine.z500 - ctrl.z500) / G).sel(lead_day=d) for d in show],
          [f"Z500 difference, day {d}" for d in show],
          contours=[ctrl.z500.sel(lead_day=d) / G for d in show],
          cbar_label="Experiment - control Z500 (m)")

**Write-up:** state your hypothesis, show one or two figures, and explain whether the model behaved as you expected. Is the response physically sensible? Would you trust it more or less than the same experiment in a physics-based model, and why?

### References
* Bi et al. (2023), Accurate medium-range global weather forecasting with 3D neural networks (Pangu-Weather), *Nature* 619, 533–538.
* Pathak et al. (2022), FourCastNet: A global data-driven high-resolution weather model using adaptive Fourier neural operators, arXiv:2202.11214.
* Weyn, Durran & Caruana (2020), Improving data-driven global weather prediction using deep convolutional neural networks on a cubed sphere, *JAMES* 12, e2020MS002109.
* Selz & Craig (2023), Can artificial intelligence-based weather prediction models simulate the butterfly effect?, *GRL* 50, e2023GL105747.
* Hakim & Masanam (2024), Dynamical tests of a deep learning weather prediction model, *AIES* 3, e230090.
* Earth2Studio documentation: https://nvidia.github.io/earth2studio